# Lab - Context Design: Whole Document or Progressive Discovery

Three tasks that give Claude a body of knowledge two ways - all of it in every request, or a table of contents and a tool to read - and let tokens and accuracy choose between them.

| Task | What you learn |
| --- | --- |
| 1 | The whole document in the system prompt: one call, every token paid |
| 2 | Progressive discovery: titles in, a read tool, tokens added over every call |
| 3 | Both strategies on a small and a large document, and a rule that chooses |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"
SYSTEM = "You are ShopAssist. Answer in one sentence."

POLICY = Path("policy.txt").read_text()       # the return policy: 16 sections
HANDBOOK = Path("handbook.txt").read_text()   # the whole support handbook: 46 sections

QUESTIONS = [   # question, a fact the right answer contains
    ("Can I return earrings?", "no"),
    ("How long does a card refund take?", "10 business days"),
    ("Who approves a refund of 250?", "human"),
    ("What happens if my return parcel is lost?", "14 days"),
    ("Do I pay customs on an international return?", "customs"),
    ("Can I exchange for another size?", "new order"),
]


def ask(**request):
    return client.messages.create(
        model=MODEL, max_tokens=300, output_config={"effort": "low"}, **request)


def text_of(response):
    return " ".join(b.text for b in response.content if b.type == "text").lower()


def chunk_by_section(text):
    """One chunk per section: its title, and the section with its title."""
    chunks = []
    for block in text.split("\n\n"):
        lines = block.strip().split("\n")
        if len(lines) >= 2:
            chunks.append({"id": lines[0].title(), "text": block.strip()})
    return chunks


def run_all(strategy, text):
    """The six questions in parallel, six at a time: tokens per question and right answers."""
    with ThreadPoolExecutor(max_workers=6) as pool:
        runs = list(pool.map(lambda q: strategy(q[0], text), QUESTIONS))
    return {"tokens_per_question": sum(used for _, used in runs) // len(QUESTIONS),
            "right": sum(expected in text_of(response)
                         for (response, _), (_, expected) in zip(runs, QUESTIONS))}

---

## Task 1 - The whole document in every request

In [ ]:
# ============================================================
# TASK 1 - The whole document in every request
# ============================================================
#
# WHAT TO DO
#   Write whole(): the document goes into the system prompt,
#   the question into the message, one call. Return the
#   response and the input tokens it took. Run the six
#   questions against the return policy.
#
# WHY IT MATTERS
#   This is the simplest design, and for a small document often
#   the right one. But every token in the prompt is paid for on
#   every call, and a long context can bury the one rule that
#   matters. The number to watch is tokens per question.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The first strategy is the whole document".
#
# HOW TO DO IT
#   Three blanks:
#
#       f"{SYSTEM}\n\n{text}"          the instruction, then the document
#       response.usage.input_tokens    what this question cost
#       POLICY                         the small document first
# ============================================================

try:
    # TODO: replace each ... below with the value named beside it
    def whole(question, text):
        response = ask(system=...,   # f"{SYSTEM}\n\n{text}"
                       messages=[{"role": "user", "content": question}])
        return response, ...   # response.usage.input_tokens
    
    policy_whole = run_all(whole, ...)   # POLICY
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print("policy, whole document:", policy_whole)

check("disc_whole", result=policy_whole, whole=whole, policy=POLICY)

---

## Task 2 - A table of contents and a tool to read

In [ ]:
# ============================================================
# TASK 2 - A table of contents and a tool to read
# ============================================================
#
# WHAT TO DO
#   Finish progressive(): Claude gets a read_section tool whose
#   input is a title from the list of all titles - the table of
#   contents, not the book. The loop runs the tool until Claude
#   answers, and adds up the input tokens of every call.
#
# WHY IT MATTERS
#   The cost of progressive discovery grows with the list of
#   titles and with what Claude actually reads, not with every
#   page you add. The price is a round trip: every tool call is
#   a second request that repeats the instructions.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Our loop runs the tool and adds up the input tokens of
#   every call".
#
# HOW TO DO IT
#   Three blanks:
#
#       list(sections)                  the titles Claude may choose from
#       response.usage.input_tokens     every call counts, not only the last
#       sections[b.input["title"]]      the section Claude asked for
# ============================================================

try:
    # TODO: replace each ... below with the value named beside it
    def progressive(question, text):
        sections = {c["id"]: c["text"] for c in chunk_by_section(text)}
        read_section = {
            "name": "read_section",
            "description": "Read one section.",
            "input_schema": {
                "type": "object", "required": ["title"],
                "properties": {"title": {"type": "string", "enum": ...}}},   # list(sections)
        }
        messages = [{"role": "user", "content": question}]
        used = 0
        while True:
            response = ask(system=SYSTEM, tools=[read_section], messages=messages)
            used += ...   # response.usage.input_tokens
            if response.stop_reason != "tool_use":
                return response, used
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": [
                {"type": "tool_result", "tool_use_id": b.id,
                 "content": ...}   # sections[b.input["title"]]
                for b in response.content if b.type == "tool_use"]})

    policy_progressive = run_all(progressive, POLICY)
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
except NameError as exc:   # provided: an earlier task has not run yet
    raise ValueError("{} - run the cells above first, Task 1 included."
                     .format(exc)) from None
print("policy, whole document:  ", policy_whole)
print("policy, progressive:     ", policy_progressive)

check("disc_progressive", result=policy_progressive, whole_result=policy_whole)

---

## Task 3 - Choose by the shape of the data

In [ ]:
# ============================================================
# TASK 3 - Choose by the shape of the data
# ============================================================
#
# WHAT TO DO
#   Run both strategies on both documents - the policy and the
#   whole handbook - and write strategy_for(): progressive
#   discovery when it saves at least a third of the tokens,
#   otherwise the whole document, cached.
#
# WHY IT MATTERS
#   The answer depends on the data, not on taste. On a small,
#   stable document the saving is small and does not pay for the
#   extra round trip - keep it in the prompt and cache it. On a
#   large or growing one the whole text costs more with every
#   page; an index and a way to read wins.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Small and stable: put it in the context and cache it".
#
# HOW TO DO IT
#   Three blanks:
#
#       run_all(progressive, text)      the second strategy, same document
#       ratio >= 1.5                    whole costs at least 1.5 times as much
#       strategy_for(row)               the choice for each document
# ============================================================

try:
    # TODO: replace each ... below with the value named beside it
    table = {}
    for name, text in [("policy", POLICY), ("handbook", HANDBOOK)]:
        table[name] = {"whole": run_all(whole, text), "progressive": ...}   # run_all(progressive, text)
    
    def strategy_for(row):
        ratio = row["whole"]["tokens_per_question"] / row["progressive"]["tokens_per_question"]
        return "progressive discovery" if ... else "whole document, cached"   # ratio >= 1.5
    
    choice = {name: ... for name, row in table.items()}   # strategy_for(row)
    if ... in choice.values():   # provided: a blank left above, said plainly
        raise TypeError("ellipsis")
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
except NameError as exc:   # provided: an earlier task has not run yet
    raise ValueError("{} - run the cells above first, Tasks 1 to 2 included."
                     .format(exc)) from None
for name, row in table.items():
    for strategy, result in row.items():
        print(f"{name:9} {strategy:12} {result['tokens_per_question']:5} tokens/question"
              f"  {result['right']}/{len(QUESTIONS)} right")
    print(f"{name:9} -> {choice[name]}")

check("disc_choice", table=table, choice=choice)

---

## Done

Measure tokens per question and accuracy for both strategies on your own documents: keep small, stable documents in a cached prompt, and switch to progressive discovery when the documents are much larger than what one answer needs.